In [1]:
import pandas as pd


In [2]:
# pick the most likely topic from dictionary / LDA
pick_topic = False
if pick_topic:
    df = pd.read_csv("merged_topic_results.csv")
    df.rename(columns={'dominant_topic': 'dominant_LDA_topic'}, inplace=True)
    cols = ['supply_side', 'demand_side', 'financial_risk', 'geopolitical_risk', 'esg_regulation', 'natural_factor']
    df['dominant_dictionary_topic'] = df[cols].idxmax(axis=1)
    df.to_csv("merged_topic_results.csv", index=False)


In [25]:

company_mentions = pd.read_csv('final company mentions.csv')
market_impact = pd.read_csv('final market impact.csv')
topic_results = pd.read_csv('merged_topic_results.csv')


In [16]:
def wide_to_long(df, value_name):
    df = df.reset_index() 
    df = df.rename(columns={'index': 'wide index'})
    return pd.melt(
        df,
        id_vars=['wide index', 'date', 'source'], 
        var_name='ticker',
        value_name=value_name
    )

# Transform both wide tables
mentions_long = wide_to_long(company_mentions, 'mentioned_times')
impact_long = wide_to_long(market_impact, 'market_impact')

# Drop meaningless zeros
mentions_long = mentions_long[mentions_long['mentioned_times'] != 0]


In [27]:
# Merge the two long tables
combined_long = pd.merge(mentions_long, impact_long[['wide index', 'ticker', 'market_impact']], 
                        on=['wide index', 'ticker'],
                        how='left')

# Merge the topic results
topic_results = topic_results.reset_index() 
topic_results = topic_results.rename(columns={'index': 'wide index'})
aimed_columns = [col for col in topic_results.columns if col not in ['date', 'source']]
combined_long = pd.merge(combined_long, topic_results[aimed_columns], 
                        on=['wide index'],
                        how='left')


In [29]:
combined_long.head()
combined_long.to_csv('long_raw_nlp_data.csv', index=False)

,wide index,date,source,ticker,mentioned_times,market_impact,supply_side,demand_side,financial_risk,geopolitical_risk,esg_regulation,natural_factor,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,dominant_LDA_topic,dominant_dictionary_topic
0,86,2024-11-01,benzinga,TUSK,1.0,1.0,3,1,5,1,0,1,0.000000,0.0,0.153179,0.819108,0.027658,3,financial_risk
1,87,2024-11-01,cision,TUSK,1.0,-1.0,3,1,5,1,0,1,0.000000,0.0,0.226684,0.719299,0.053912,3,financial_risk
2,88,2024-07-20,seekingalpha,TUSK,1.0,0.0,2,0,2,0,0,1,0.203023,0.0,0.091212,0.705679,0.000000,3,supply_side
3,4106,2015-05-15,Thomson_Reuters,TUSK,1.0,1.0,0,0,0,1,0,1,0.133783,0.0,0.838352,0.000000,0.027304,2,geopolitical_risk
4,4107,2018-09-28,Thomson_Reuters,TUSK,1.0,0.0,0,0,0,0,0,1,0.000000,0.0,0.981171,0.000000,0.000000,2,natural_factor


In [32]:
# Create quarterly aggregated data
def create_quarterly_features(df):
    # Ensure mentioned_times is integer
    df['mentioned_times'] = df['mentioned_times'].fillna(0).astype(int)
    
    # Ensure market_impact is filled (if needed，可以改成dropna)
    df['market_impact'] = df['market_impact'].fillna(0)

    # Add quarter column
    df['quarter'] = pd.to_datetime(df['date']).dt.to_period('Q')

    # Group by ticker and quarter
    grouped = df.groupby(['ticker', 'quarter'])

    # Calculate basic statistics
    quarterly_stats = grouped.agg({
        'mentioned_times': 'sum',
        'market_impact': ['mean', 'sum']
    }).reset_index()

    # Rename columns
    quarterly_stats.columns = ['ticker', 'quarter', 
                                'total_mentions', 
                                'avg_market_impact',
                                'sum_market_impact']

    # Calculate total mentions per quarter across all companies
    total_mentions_by_quarter = df.groupby('quarter')['mentioned_times'].sum().reset_index()
    total_mentions_by_quarter.columns = ['quarter', 'quarter_total_mentions']

    # Merge total mentions
    quarterly_stats = pd.merge(quarterly_stats, total_mentions_by_quarter,
                               on='quarter', how='left')

    # Calculate mention percentage
    quarterly_stats['mention_percentage'] = quarterly_stats['total_mentions'] / quarterly_stats['quarter_total_mentions']

    # Calculate weighted market impact
    quarterly_stats['weighted_market_impact'] = quarterly_stats['sum_market_impact'] * quarterly_stats['mention_percentage']

    return quarterly_stats


def calculate_by_topic(df, topic_col):
    """
    Calculate quarterly features by topic and ticker, normalized by total mentions per quarter (not per topic).
    
    Parameters:
    - df: DataFrame containing at least ['date', 'ticker', 'mentioned_times', 'market_impact', topic_col]
    - topic_col: str, the name of the topic column (e.g., 'dominant_dictionary_topic' or 'dominant_LDA_topic')
    
    Returns:
    - DataFrame with quarterly statistics by topic and ticker
    """
    
    # Ensure mentioned_times and market_impact are valid
    df['mentioned_times'] = df['mentioned_times'].fillna(0).astype(int)
    df['market_impact'] = df['market_impact'].fillna(0)

    # Add quarter column
    df['quarter'] = pd.to_datetime(df['date']).dt.to_period('Q')

    # Group by topic, ticker, and quarter
    grouped = df.groupby([topic_col, 'ticker', 'quarter'])

    # Basic aggregations
    topic_stats = grouped.agg({
        'mentioned_times': 'sum',
        'market_impact': ['mean', 'sum']
    }).reset_index()

    # Rename columns for clarity
    topic_stats.columns = [topic_col, 'ticker', 'quarter',
                            'total_mentions',
                            'avg_market_impact',
                            'sum_market_impact']

    # Calculate total mentions per quarter (across all topics and tickers)
    total_mentions_by_quarter = df.groupby('quarter')['mentioned_times'].sum().reset_index()
    total_mentions_by_quarter.columns = ['quarter', 'quarter_total_mentions']

    # Merge total quarterly mentions
    topic_stats = pd.merge(topic_stats, total_mentions_by_quarter,
                            on='quarter', how='left')

    # Calculate mention percentage (per ticker-topic vs total mentions in the quarter)
    topic_stats['mention_percentage'] = topic_stats['total_mentions'] / topic_stats['quarter_total_mentions']

    # Calculate weighted market impact
    topic_stats['weighted_market_impact'] = topic_stats['sum_market_impact'] * topic_stats['mention_percentage']

    return topic_stats


# Calculate quarterly aggregated data
quarterly_data = create_quarterly_features(combined_long)
quarterly_data.to_csv('quarterly_company_stats.csv', index=False)

# By dictionary topic
dictionary_topic_stats = calculate_by_topic(combined_long, topic_col='dominant_dictionary_topic')
dictionary_topic_stats.to_csv('quarterly_dictionary_topic_stats.csv', index=False)

# By LDA topic
lda_topic_stats = calculate_by_topic(combined_long, topic_col='dominant_LDA_topic')
lda_topic_stats.to_csv('quarterly_lda_topic_stats.csv', index=False)


In [1]:
# If running from this step, load the data
try:
    combined_long
except NameError:
    import pandas as pd
    combined_long = pd.read_csv('long_raw_nlp_data.csv')

In [7]:
def calculate_quarterly_topic_features(df):
    """
    Calculate quarterly features without grouping by ticker.
    Includes:
    - Overall (no topic)
    - By dictionary topic
    - By LDA topic

    Parameters:
    - df: DataFrame containing ['date', 'mentioned_times', 'market_impact', 'dominant_dictionary_topic', 'dominant_LDA_topic']

    Returns:
    - DataFrame with quarterly features for each topic type
    """

    # Ensure mentioned_times and market_impact are valid
    df['mentioned_times'] = df['mentioned_times'].fillna(0).astype(int)
    df['market_impact'] = df['market_impact'].fillna(0)

    # Add quarter column
    df['quarter'] = pd.to_datetime(df['date']).dt.to_period('Q')

    results = []

    for topic_type, topic_col in [('all', None), 
                                  ('dictionary', 'dominant_dictionary_topic'), 
                                  ('lda', 'dominant_LDA_topic')]:
        
        if topic_col is None:
            temp = df.copy()
            temp['topic'] = 'all'
        else:
            temp = df.rename(columns={topic_col: 'topic'})

        # Group by topic and quarter
        grouped = temp.groupby(['topic', 'quarter'])

        stats = grouped.agg({
            'mentioned_times': 'sum',
            'market_impact': ['mean', 'sum']
        }).reset_index()

        stats.columns = ['topic', 'quarter',
                         'total_mentions',
                         'avg_market_impact',
                         'sum_market_impact']

        # Calculate total mentions per quarter (for normalization)
        total_mentions_by_quarter = temp.groupby('quarter')['mentioned_times'].sum().reset_index()
        total_mentions_by_quarter.columns = ['quarter', 'quarter_total_mentions']

        # Merge
        stats = pd.merge(stats, total_mentions_by_quarter, on='quarter', how='left')

        # Calculate percentage and weighted market impact
        stats['mention_percentage'] = stats['total_mentions'] / stats['quarter_total_mentions']
        stats['weighted_market_impact'] = stats['sum_market_impact'] * stats['mention_percentage']

        # Add topic type label
        stats['topic_type'] = topic_type

        results.append(stats)

    # Combine all
    final_result = pd.concat(results, ignore_index=True)

    return final_result

quarterly_features = calculate_quarterly_topic_features(combined_long)

# Filter the rows where the year is greater than 2010
quarterly_features['year'] = quarterly_features['quarter'].dt.year
quarterly_features = quarterly_features[quarterly_features['year'] > 2010]
quarterly_features.drop(columns=['year'], inplace=True)
quarterly_features.to_csv('quarterly_topic_stats.csv', index=False)

